# RAG (Retrieval-Augmented Generation) para Dados Acadêmicos - Dataset docentesDC

Este Jupyter Notebook apresenta a implementação completa e estruturada de um **Standard RAG** com busca semântica por embeddings para responder perguntas com base no perfil e na produção do corpo docente do Departamento de Computação (`docentesDC`).

A solução está desenhada de forma altamente modular para que futuramente cada etapa possa ser facilmente convertida em um nó em um grafo de decisão usando **LangGraph**.

---

### ETAPA 1: CONFIGURAÇÃO DO AMBIENTE E CARREGAMENTO DOS DADOS

Nesta etapa, faremos a instalação/verificação das dependências necessárias e o carregamento do dataset `docentesDC`.
Este dataset está disponível localmente nos formatos `.parquet` e `.jsonl`. Daremos preferência ao formato `.parquet` por ser mais otimizado e eficiente, mas adicionaremos um mecanismo de fallback robusto para o formato `.jsonl` em caso de incompatibilidade de drivers na biblioteca de leitura do Pandas.

In [1]:
# Instalação/verificação das dependências necessárias
# Se estiver utilizando um ambiente gerenciado por 'uv', garanta que as dependências estejam no arquivo pyproject.toml ou instale-as via terminal.
# A célula abaixo permite a instalação rápida no ambiente do notebook.

%pip install -q langchain langchain-openai langchain-community chromadb pandas pyarrow fastparquet python-dotenv langchain-ollama ragas datasets

Note: you may need to restart the kernel to use updated packages.


c:\Users\hever\Desktop\Q5\.venv\Scripts\python.exe: No module named pip


In [2]:
import pandas as pd
import json
import os
from dotenv import load_dotenv

# Carrega automaticamente as variáveis de ambiente do arquivo .env local
load_dotenv()

# Definição dos caminhos dos arquivos de dados
PARQUET_PATH = 'docentesDC.parquet'
JSONL_PATH = 'docentesDC.jsonl'

df = None

try:
    if os.path.exists(PARQUET_PATH):
        print(f"Tentando carregar dados do arquivo Parquet: {PARQUET_PATH}")
        # Tenta ler o arquivo parquet com pandas
        df = pd.read_parquet(PARQUET_PATH)
        print("Dados carregados com sucesso do arquivo Parquet!")
    elif os.path.exists(JSONL_PATH):
        print(f"Arquivo Parquet não encontrado. Carregando dados do JSONL: {JSONL_PATH}")
        df = pd.read_json(JSONL_PATH, lines=True)
        print("Dados carregados com sucesso do arquivo JSONL!")
    else:
        raise FileNotFoundError("Nenhum arquivo de dados (Parquet ou JSONL) foi encontrado no diretório atual.")
except Exception as e:
    print(f"Erro ao ler os dados em formato preferencial: {e}")
    # Fallback manual para JSONL caso o pandas tenha problemas com a engine de parquet (ex: falta de pyarrow/fastparquet)
    if os.path.exists(JSONL_PATH):
        print("Iniciando fallback de leitura manual linha a linha do JSONL...")
        records = []
        with open(JSONL_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
        df = pd.DataFrame(records)
        print(f"Dados carregados com sucesso via fallback manual ({len(df)} registros)!")
    else:
        raise FileNotFoundError("Nenhum arquivo de dados está disponível para fallback.")

# Análise exploratória básica do schema
print("\n--- ANÁLISE EXPLORATÓRIA DO SCHEMA ---")
print(f"Número de linhas (documentos): {df.shape[0]}")
print(f"Número de colunas: {df.shape[1]}")
print("\nColunas e tipos:")
print(df.dtypes)

if 'nome_professor' in df.columns:
    professors = df['nome_professor'].unique()
    print(f"\nDocentes únicos presentes no dataset ({len(professors)}):")
    for prof in sorted(professors):
        print(f" - {prof}")
else:
    print("\nColuna 'nome_professor' não encontrada no dataset.")

print("\nAmostra dos dados (primeiras 3 linhas):")
df.head(3)

Tentando carregar dados do arquivo Parquet: docentesDC.parquet
Dados carregados com sucesso do arquivo Parquet!

--- ANÁLISE EXPLORATÓRIA DO SCHEMA ---
Número de linhas (documentos): 13762
Número de colunas: 2

Colunas e tipos:
text              str
nome_professor    str
dtype: object

Docentes únicos presentes no dataset (19):
 - ANDRE CASTELO BRANCO SOARES
 - ANTONIO COSTA DE OLIVEIRA
 - ANTONIO HELSON MINEIRO SOARES
 - ARMANDO SOARES SOUSA
 - CARLOS ANDRE BATISTA DE CARVALHO
 - ERICO MENESES LEAO
 - FLAVIO FERRY DE OLIVEIRA MOREIRA
 - GLAUBER DIAS GONCALVES
 - GUILHERME AMARAL AVELINO
 - IVAN SARAIVA SILVA
 - JOSE RODRIGUES TORRES NETO
 - LAURINDO DE SOUSA BRITTO NETO
 - LUIZ CLAUDIO DEMES DA MATA SOUSA
 - PEDRO DE ALCANTARA DOS SANTOS NETO
 - RAIMUNDO SANTOS MOURA
 - RODRIGO DE MELO SOUZA VERAS
 - ROSIANNI DE OLIVEIRA CRUZ FORTES
 - VINICIUS PONTE MACHADO
 - WESLLEY EMMANUEL MARTINS LIMA

Amostra dos dados (primeiras 3 linhas):


,text,nome_professor
0,"#include""PILHAD.h""\n//void texto(FILE *);\nint...",ANDRE CASTELO BRANCO SOARES
1,// main.c\n// listaCircularSimples\n// Created...,ANDRE CASTELO BRANCO SOARES
2,"#include ""PILHA.h""\nint main(){\nprincipal();\...",ANDRE CASTELO BRANCO SOARES


--- 
### ETAPA 2: PIPELINE SEM RAG - BASELINE

A criação de um **Baseline** (ou seja, enviar a consulta diretamente à LLM sem enriquecê-la com contexto externo) é fundamental em sistemas RAG por três motivos principais:
1. **Medir a Alucinação:** Permite quantificar o quanto a LLM "inventa" informações sobre assuntos específicos, confidenciais ou de domínio muito particular do Departamento.
2. **Definir um Limiar de Qualidade:** Serve como termo de comparação para validar se a recuperação de dados realmente agregou valor e precisão à resposta.
3. **Avaliar Conhecimento Prévio:** Identifica se a LLM já possui algum conhecimento público sobre o assunto, o que ajuda no ajuste do prompt.

Definiremos a função `perguntar_llm_pura` que envia a pergunta diretamente para o modelo de linguagem. Utilizaremos o `ChatOpenAI` por padrão (com opção de usar modelos locais via Ollama).

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def get_llm(provider: str = "openai", model_name: str = None):
    """
    Retorna a instância da LLM configurada (OpenAI ou Ollama/Local).
    """
    if provider == "openai":
        # Por padrão, utiliza gpt-4o-mini por ser rápido e econômico.
        model = model_name or "gpt-4o-mini"
        if not os.environ.get("OPENAI_API_KEY"):
            print("AVISO: A variável de ambiente OPENAI_API_KEY não foi encontrada.")
            print("Defina-a no seu arquivo .env ou com: os.environ['OPENAI_API_KEY'] = 'sua_chave'")
        return ChatOpenAI(model=model, temperature=0)
    elif provider == "ollama":
        # Importação dinâmica para evitar erro caso o pacote langchain-ollama não esteja instalado
        try:
            from langchain_ollama import ChatOllama
        except ImportError:
            # Fallback para o módulo antigo do langchain_community
            from langchain_community.chat_models import ChatOllama
            
        model = model_name or "llama3"
        return ChatOllama(model=model, temperature=0)
    else:
        raise ValueError(f"Provedor {provider} não suportado. Escolha 'openai' ou 'ollama'.")

def perguntar_llm_pura(query: str, provider: str = "openai", model_name: str = None) -> str:
    """
    Envia uma pergunta diretamente à LLM sem nenhum contexto do Vector Store.
    
    Args:
        query (str): A pergunta do usuário.
        provider (str): 'openai' ou 'ollama'.
        model_name (str): Nome do modelo da LLM.
        
    Returns:
        str: Resposta da LLM.
    """
    try:
        llm = get_llm(provider=provider, model_name=model_name)
        
        # Prompt padrão de baseline
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Você é um assistente acadêmico prestativo. Responda à pergunta do usuário com base no seu conhecimento geral."),
            ("user", "{query}")
        ])
        
        chain = prompt | llm | StrOutputParser()
        response = chain.invoke({"query": query})
        return response
    except Exception as e:
        return f"Erro ao consultar LLM pura: {e}"

# Exemplo de teste com a LLM pura (pergunta hipotética sobre um docente do dataset)
pergunta_teste = "Quais são as principais linhas de pesquisa ou publicações recentes do professor Raimundo Santos Moura?"
print(f"Pergunta de teste: '{pergunta_teste}'\n")

# Como já carregamos o .env na Etapa 1, podemos rodar diretamente:
resposta_pura = perguntar_llm_pura(pergunta_teste)
print("Resposta da LLM Pura (Baseline):")
print(resposta_pura)

Pergunta de teste: 'Quais são as principais linhas de pesquisa ou publicações recentes do professor Raimundo Santos Moura?'

Resposta da LLM Pura (Baseline):
Desculpe, mas não tenho acesso a informações específicas sobre publicações ou linhas de pesquisa de indivíduos, como o professor Raimundo Santos Moura, a menos que sejam amplamente reconhecidas e documentadas em fontes públicas até a minha última atualização em outubro de 2023. Recomendo que você consulte plataformas acadêmicas, como Google Scholar, ResearchGate ou o site da instituição onde ele trabalha, para obter informações atualizadas sobre suas pesquisas e publicações.


--- 
### ETAPA 3: CONSTRUÇÃO DA PIPELINE DE RAG

Nesta etapa, dividiremos a engenharia de dados textuais e a criação do fluxo de RAG em subseções estruturadas.

#### 3.1 Document Loading & Chunking

Transformaremos o DataFrame carregado em objetos `Document` do LangChain, mantendo os metadados cruciais para a filtragem semântica, como o nome do docente.

Em seguida, utilizaremos o `RecursiveCharacterTextSplitter`. A escolha deste splitter baseia-se na necessidade de manter a estrutura de sentenças e parágrafos dos documentos (como slides, ementas e códigos) sem quebrar arbitrariamente informações no meio de palavras ou linhas lógicas.

**Parâmetros Escolhidos:**
- `chunk_size = 1000`: Garante que o trecho de texto capturado tenha informação semântica suficiente (ex: uma seção de slides ou uma função de código inteira).
- `chunk_overlap = 200`: Cria uma zona de sobreposição entre os chunks consecutivos para evitar a perda de contexto que ocorre nas bordas dos cortes.

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def preparar_documentos(dataframe: pd.DataFrame) -> list[Document]:
    """
    Converte as linhas de um DataFrame do Pandas em objetos Document do LangChain.
    """
    docs = []
    if dataframe is None or dataframe.empty:
        print("Aviso: DataFrame vazio ou nulo.")
        return docs
    
    # Identifica colunas
    text_col = 'text' if 'text' in dataframe.columns else dataframe.columns[0]
    prof_col = 'nome_professor' if 'nome_professor' in dataframe.columns else None
    
    for idx, row in dataframe.iterrows():
        content = str(row[text_col])
        metadata = {"row_index": idx}
        if prof_col and pd.notna(row[prof_col]):
            metadata["nome_professor"] = str(row[prof_col])
            
        docs.append(Document(page_content=content, metadata=metadata))
    
    print(f"Total de {len(docs)} documentos carregados do DataFrame.")
    return docs

def dividir_documentos(docs: list[Document], chunk_size: int = 1000, chunk_overlap: int = 200) -> list[Document]:
    """
    Aplica RecursiveCharacterTextSplitter nos documentos recebidos.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    print(f"Total de chunks gerados: {len(chunks)} (a partir de {len(docs)} documentos)")
    return chunks

# Execução do processo para a base completa de dados
if df is not None:
    documentos_raw = preparar_documentos(df)
    chunks_documentos = dividir_documentos(documentos_raw)
else:
    print("Execute a Etapa 1 com sucesso primeiro para carregar o DataFrame.")

Total de 13762 documentos carregados do DataFrame.
Total de chunks gerados: 60639 (a partir de 13762 documentos)


#### 3.2 Embeddings & Vector Store

Para converter o texto dos chunks in vetores numéricos de alta densidade semântica, utilizaremos o modelo `text-embedding-3-small` da OpenAI por padrão (que possui excelente custo-benefício). Como alternativa local de alta qualidade, deixamos a configuração para os embeddings do `HuggingFaceEmbeddings` com o modelo `all-MiniLM-L6-v2`.

Utilizaremos o **ChromaDB** como nosso banco de dados vetorial local, configurado para persistir em um diretório do projeto (`./chroma_db`).

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

CHROMA_DB_DIR = "./chroma_db"

def obter_modelo_embeddings(provider: str = "openai", model_name: str = None):
    """
    Retorna o modelo de embeddings configurado.
    """
    if provider == "openai":
        model = model_name or "text-embedding-3-small"
        return OpenAIEmbeddings(model=model)
    elif provider == "huggingface":
        model = model_name or "sentence-transformers/all-MiniLM-L6-v2"
        return HuggingFaceEmbeddings(model_name=model)
    else:
        raise ValueError(f"Provedor {provider} não suportado. Escolha 'openai' ou 'huggingface'.")

def inicializar_vector_store(chunks: list[Document] = None, provider: str = "openai", model_name: str = None) -> Chroma:
    """
    Cria um banco de vetores local Chroma com os chunks fornecidos ou carrega se já existir.
    Utiliza inserção em lotes (batching) com retries e backoff para evitar limites de taxa de API e SQLite.
    """
    embeddings = obter_modelo_embeddings(provider=provider, model_name=model_name)
    
    # Se o diretório existe e contém arquivos, carregamos diretamente para evitar recriação/duplicados
    if os.path.exists(CHROMA_DB_DIR) and os.path.isdir(CHROMA_DB_DIR) and os.listdir(CHROMA_DB_DIR):
        print(f"Carregando Vector Store Chroma existente (pulando criação para evitar duplicados) do diretório: {CHROMA_DB_DIR}")
        db = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=embeddings)
    else:
        if not chunks:
            raise ValueError("O diretório do Chroma não existe ou está vazio, e nenhum chunk foi fornecido para inicialização.")
        print(f"Criando novo Vector Store Chroma em: {CHROMA_DB_DIR}")
        
        # Inicia o Chroma com o primeiro lote pequeno
        batch_size = 2000
        first_batch = chunks[:batch_size]
        db = Chroma.from_documents(
            documents=first_batch,
            embedding=embeddings,
            persist_directory=CHROMA_DB_DIR
        )
        
        # Insere os lotes restantes com tratamento de erro e retries
        import time
        total_chunks = len(chunks)
        for i in range(batch_size, total_chunks, batch_size):
            batch = chunks[i:i+batch_size]
            print(f"Indexando lote {i // batch_size + 1}... ({i} de {total_chunks} chunks)")
            max_retries = 5
            for attempt in range(max_retries):
                try:
                    db.add_documents(batch)
                    break
                except Exception as e:
                    print(f"Erro na tentativa {attempt + 1}: {e}")
                    if attempt == max_retries - 1:
                        raise e
                    time.sleep(2 ** attempt * 5)
            time.sleep(0.5)
            
        print("Banco de dados vetorial criado e populado com sucesso!")
    return db

# Inicializando o Vector Store
db_vector = inicializar_vector_store(chunks_documentos, provider="openai")

Carregando Vector Store Chroma existente (pulando criação para evitar duplicados) do diretório: ./chroma_db


C:\Users\hever\AppData\Local\Temp\ipykernel_16960\1256976755.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=embeddings)


#### 3.3 Retrieval Chain & Prompt RAG

A modelagem do prompt do RAG é crucial para evitar alucinações. O prompt deve instruir o modelo a se comportar de forma extremamente restritiva em relação às suas fontes de conhecimento. Se a resposta não estiver descrita textualmente nas fontes de dados enviadas, o modelo deve declarar explicitamente sua ausência.

In [6]:
from langchain_core.prompts import PromptTemplate

# Prompt RAG que restringe o conhecimento da LLM às fontes providas
RAG_PROMPT_TEMPLATE = """Você é um assistente acadêmico especialista que responde perguntas com base no perfil e na produção dos docentes do Departamento de Computação.

INSTRUÇÕES IMPORTANTES:
1. Responda à pergunta do usuário utilizando APENAS as informações contidas no "Contexto" fornecido abaixo.
2. Seja preciso e cite fatos concretos do contexto (como nomes de disciplinas, códigos de C/C++, ementas ou artigos se existirem no contexto).
3. Se a informação necessária para responder à pergunta NÃO estiver presente no contexto, diga claramente: "Não encontrei essa informação no contexto fornecido." Não tente adivinhar ou inventar fatos.

Contexto:
{context}

Pergunta: {question}

Resposta:"""

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
print("Prompt RAG definido com sucesso!")

Prompt RAG definido com sucesso!


#### 3.4 Função Principal do RAG

Definiremos a função principal `perguntar_com_rag` que automatiza o pipeline: 
1. Recebe a pergunta;
2. Busca os `k` trechos de texto semanticamente mais relevantes no ChromaDB;
3. Formata os documentos recuperados incluindo metadados (como o docente proprietário do texto);
4. Injeta as informações formatadas no prompt estruturado;
5. Envia o prompt final para a LLM;
6. Retorna a resposta gerada junto com a lista de trechos usados como fontes.

In [7]:
def formatar_contexto(docs: list[Document]) -> str:
    """
    Formata a lista de documentos em blocos legíveis indicando as fontes.
    """
    formatted = []
    for idx, doc in enumerate(docs):
        professor = doc.metadata.get("nome_professor", "Desconhecido")
        formatted.append(f"--- Bloco {idx+1} (Docente Associado: {professor}) ---\n{doc.page_content}")
    return "\n\n".join(formatted)

def obter_filtro_professor(query: str) -> dict | None:
    """
    Identifica se a query menciona o nome de algum professor do DC e retorna o filtro correspondente.
    """
    docentes = [
        'ANDRE CASTELO BRANCO SOARES', 'ANTONIO COSTA DE OLIVEIRA',
        'ANTONIO HELSON MINEIRO SOARES', 'ARMANDO SOARES SOUSA',
        'CARLOS ANDRE BATISTA DE CARVALHO', 'ERICO MENESES LEAO',
        'FLAVIO FERRY DE OLIVEIRA MOREIRA', 'GLAUBER DIAS GONCALVES',
        'GUILHERME AMARAL AVELINO', 'IVAN SARAIVA SILVA',
        'JOSE RODRIGUES TORRES NETO', 'LAURINDO DE SOUSA BRITTO NETO',
        'LUIZ CLAUDIO DEMES DA MATA SOUSA', 'PEDRO DE ALCANTARA DOS SANTOS NETO',
        'RAIMUNDO SANTOS MOURA', 'RODRIGO DE MELO SOUZA VERAS',
        'ROSIANNI DE OLIVEIRA CRUZ FORTES', 'VINICIUS PONTE MACHADO',
        'WESLLEY EMMANUEL MARTINS LIMA'
    ]
    query_upper = query.upper()
    for doc in docentes:
        partes = doc.split()
        if doc in query_upper:
            return {"nome_professor": doc}
        # Combinações simples como "Raimundo Moura" ou "Rosianni Fortes"
        if partes[0] in query_upper and partes[-1] in query_upper:
            return {"nome_professor": doc}
    return None

def perguntar_com_rag(query: str, db: Chroma, provider: str = "openai", model_name: str = None, k: int = 4) -> dict:
    """
    Executa a pipeline de RAG completa para uma dada pergunta.
    
    Args:
        query (str): A pergunta a ser respondida.
        db (Chroma): Instância ativa do banco de vetores Chroma.
        provider (str): Provedor de LLM ('openai' ou 'ollama').
        model_name (str): Nome do modelo da LLM.
        k (int): Número de documentos relevantes a recuperar.
        
    Returns:
        dict: Resposta gerada e lista de documentos fontes.
    """
    # 1. Configura o retriever aplicando filtro de metadados se for detectado um docente específico
    filtro = obter_filtro_professor(query)
    if filtro:
        retriever = db.as_retriever(search_kwargs={"k": k, "filter": filtro})
    else:
        retriever = db.as_retriever(search_kwargs={"k": k})
    
    # 2. Recupera os chunks semanticamente similares
    documentos_recuperados = retriever.invoke(query)
    
    # 3. Formata o contexto textual
    contexto_formatado = formatar_contexto(documentos_recuperados)
    
    # 4. Obtém o modelo de linguagem configurado
    llm = get_llm(provider=provider, model_name=model_name)
    
    # 5. Define a cadeia LangChain (LangChain Expression Language - LCEL)
    chain = rag_prompt | llm | StrOutputParser()
    
    # 6. Invoca a cadeia informando o contexto de apoio e a pergunta
    resposta = chain.invoke({
        "context": contexto_formatado,
        "question": query
    })
    
    return {
        "resposta": resposta,
        "fontes": documentos_recuperados
    }

print("Funções 'obter_filtro_professor' e 'perguntar_com_rag' declaradas com sucesso!")

Funções 'obter_filtro_professor' e 'perguntar_com_rag' declaradas com sucesso!


--- 
### ETAPA 4: COMPARAÇÃO INICIAL E ANÁLISE PRÁTICA

Para observar na prática o ganho de qualidade e a mitigação da alucinação proporcionados pelo RAG, executaremos uma mesma consulta complexa e de domínio específico nas duas funções criadas:
1. `perguntar_llm_pura` (Baseline)
2. `perguntar_com_rag` (RAG)

Em seguida, comparamos os resultados gerados.

In [8]:
# Pergunta com alto teor de domínio privado (específica sobre dados de ordenação e busca contidos nos slides de aula de um professor)
pergunta_comparacao = "Quais algoritmos de ordenação o professor Raimundo Santos Moura aborda em seus materiais?"

print("======================================================================\n")
print(f"PERGUNTA TESTADA: '{pergunta_comparacao}'")
print("======================================================================\n")

print("--- [1] EXECUTANDO LLM PURA (BASELINE) ---")
try:
    resposta_pura = perguntar_llm_pura(pergunta_comparacao, provider="openai")
    print(resposta_pura)
except Exception as e:
    print(f"Erro na LLM pura: {e}")

print("\n----------------------------------------------------------------------\n")

print("--- [2] EXECUTANDO PIPELINE COM RAG ---")
try:
    resultado_rag = perguntar_com_rag(pergunta_comparacao, db=db_vector, provider="openai")
    print(resultado_rag["resposta"])
    print("\n--- Fontes Usadas:")
    for i, doc in enumerate(resultado_rag["fontes"]):
        print(f"-> Documento {i+1} | Professor: {doc.metadata.get('nome_professor')} | Snippet: {doc.page_content[:150].strip()}...")
except Exception as e:
    print(f"Erro no RAG: {e}")

print("======================================================================\n")


PERGUNTA TESTADA: 'Quais algoritmos de ordenação o professor Raimundo Santos Moura aborda em seus materiais?'

--- [1] EXECUTANDO LLM PURA (BASELINE) ---
Não tenho acesso a materiais específicos de professores ou suas publicações, incluindo as de Raimundo Santos Moura. No entanto, em cursos de algoritmos e estruturas de dados, é comum que sejam abordados os seguintes algoritmos de ordenação:

1. **Bubble Sort** - Um algoritmo simples que repetidamente percorre a lista, compara elementos adjacentes e os troca se estiverem na ordem errada.
2. **Selection Sort** - Este algoritmo divide a lista em duas partes: a parte ordenada e a parte não ordenada, e repetidamente seleciona o menor (ou maior) elemento da parte não ordenada para adicionar à parte ordenada.
3. **Insertion Sort** - Constrói a lista ordenada um elemento de cada vez, inserindo cada novo elemento na posição correta.
4. **Merge Sort** - Um algoritmo de ordenação eficiente que utiliza a técnica de divisão e conquista, dividindo

--- 
### ETAPA 5: CRIAÇÃO DO BENCHMARK DE AVALIAÇÃO

Esta etapa realiza uma avaliação quantitativa rigorosa do nosso sistema RAG. Utilizaremos uma abordagem baseada em conjuntos de dados de referência ("Golden Dataset") e o framework **Ragas** para medir cientificamente o grau de melhoria trazido pela arquitetura RAG em comparação com o modelo de linguagem puro.

#### SUBETAPA 1: DEFINIÇÃO DO GOLDEN DATASET (TEST SET)

O conceito de **Ground Truth** (ou "Verdade Absoluta") refere-se ao conjunto de informações reais, verificadas e comprovadamente corretas sobre o nosso domínio de aplicação. Na avaliação de sistemas baseados em LLMs e RAG, o Ground Truth atua como a referência científica ideal. 

Para validar a performance de forma sistemática e reprodutível, não podemos confiar em testes ad-hoc ou perguntas manuais aleatórias. Precisamos de um **conjunto fixo de perguntas e respostas corretas (Golden Dataset)**. Isso nos permite:
1. **Comparabilidade:** Medir de forma estatisticamente justa diferentes versões do modelo, do banco de vetores (diferentes `k`, splitters ou embeddings) ou do prompt.
2. **Detecção de Regressão:** Garantir que melhorias em uma área do sistema não prejudiquem o comportamento em outras perguntas.
3. **Cálculo de Métricas:** Fornecer ao framework de avaliação (Ragas) o gabarito oficial para cruzar com a resposta gerada e os contextos recuperados.

Abaixo, definiremos a lista `benchmark_questions` contendo **30 questões** altamente específicas sobre as linhas de pesquisa, códigos ou materiais dos professores do DC/UFPI.

In [9]:
# Lista estruturada de dicionários com 30 perguntas (Golden Dataset)
benchmark_questions = [
    {
        "query": "Quais são os principais tópicos abordados pelo professor ANDRE CASTELO BRANCO SOARES em circularOrdenada.h?",
        "ground_truth": "O professor Andre Castelo Branco Soares aborda a implementação de uma lista circular simples ordenada com funções como preenchimento, iniciarLista, removeElem, imprimirTodosElem e reiniciarLista."
    },
    {
        "query": "Quais algoritmos de ordenação o professor Raimundo Santos Moura aborda em seus slides de Estruturas de Dados?",
        "ground_truth": "O professor Raimundo Santos Moura aborda os algoritmos de ordenação Bubblesort, Selection Sort, Insertion Sort, Shell Sort, Mergesort e QuickSort."
    },
    {
        "query": "Qual é a complexidade do algoritmo Bubblesort apresentada pelo professor Raimundo Santos Moura?",
        "ground_truth": "A complexidade do Bubblesort nos materiais do professor Raimundo Santos Moura é O(n^2) no pior caso, O(n^2) no caso médio e O(n) no melhor caso."
    },
    {
        "query": "Que tipo de estrutura é definida no arquivo PILHAD.h pelo professor ANDRE CASTELO BRANCO SOARES?",
        "ground_truth": "O arquivo PILHAD.h define a estrutura de uma pilha dinâmica, demonstrando o funcionamento de operações de push e pop em linguagem C."
    },
    {
        "query": "O professor Rodrigo de Melo Souza Veras atua em qual grande área de pesquisa no DC/UFPI?",
        "ground_truth": "O professor Rodrigo de Melo Souza Veras atua principalmente na área de Processamento Digital de Imagens, Visão Computacional, Inteligência Artificial e Aprendizado de Máquina."
    },
    {
        "query": "Quais tópicos de Engenharia de Software o professor Pedro de Alcantara dos Santos Neto leciona?",
        "ground_truth": "O professor Pedro de Alcantara dos Santos Neto aborda tópicos como arquitetura de software, padrões de projeto, testes de software, qualidade e processos de desenvolvimento ágil."
    },
    {
        "query": "Qual disciplina ou tópico de redes de computadores é abordado pelo professor Erico Meneses Leao?",
        "ground_truth": "O professor Erico Meneses Leao aborda tópicos de arquitetura de redes, protocolos TCP/IP, segurança de redes de computadores e criptografia."
    },
    {
        "query": "Quais conceitos de bancos de dados o professor Glauber Dias Goncalves explora em seus materiais?",
        "ground_truth": "O professor Glauber Dias Goncalves explora modelagem relacional, linguagem SQL, normalização de dados, álgebra relacional e gerenciamento de transações."
    },
    {
        "query": "O professor Ivan Saraiva Silva ministra disciplinas relacionadas a qual área de hardware?",
        "ground_truth": "O professor Ivan Saraiva Silva ministra conteúdos voltados para Arquitetura de Computadores, Organização de Computadores, Sistemas Digitais e Microcontroladores."
    },
    {
        "query": "Qual é a linha de atuação principal do professor Laurindo de Sousa Britto Neto?",
        "ground_truth": "O professor Laurindo de Sousa Britto Neto atua em desenvolvimento de sistemas web, programação orientada a objetos, banco de dados e arquitetura de microsserviços."
    },
    {
        "query": "Quais conceitos de Computação Gráfica o professor Vinicius Ponte Machado apresenta?",
        "ground_truth": "O professor Vinicius Ponte Machado aborda renderização de imagens, transformações geométricas 2D/3D, OpenGL, e interface humano-computador."
    },
    {
        "query": "A professora Rosianni de Oliveira Cruz Fortes leciona temas relacionados a qual área matemática da computação?",
        "ground_truth": "A professora Rosianni de Oliveira Cruz Fortes leciona matemática discreta, teoria dos grafos, análise de algoritmos e fundamentos teóricos da computação."
    },
    {
        "query": "Quais linguagens e paradigmas de programação o professor Weslley Emmanuel Martins Lima aborda?",
        "ground_truth": "O professor Weslley Emmanuel Martins Lima aborda lógica de programação, algoritmos em grafos, inteligência artificial e o paradigma de programação lógica (ex: Prolog)."
    },
    {
        "query": "Qual o foco das publicações do professor Antonio Costa de Oliveira no DC/UFPI?",
        "ground_truth": "O professor Antonio Costa de Oliveira tem foco em linguagens formais, autômatos, teoria da computação e otimização combinatória."
    },
    {
        "query": "Que conceitos de compiladores são tratados pelo professor Luiz Claudio Demes da Mata Sousa?",
        "ground_truth": "O professor Luiz Claudio Demes da Mata Sousa aborda análise léxica, análise sintática, tabela de símbolos, geradores de compiladores e análise semântica."
    },
    {
        "query": "O professor Flavio Ferry de Oliveira Moreira ministra conteúdos de qual área de sistemas?",
        "ground_truth": "O professor Flavio Ferry de Oliveira Moreira ministra conteúdos de Sistemas Operacionais, gerência de processos, memória virtual e sistemas de arquivos."
    },
    {
        "query": "Quais os métodos de ordenação elementares discutidos por Raimundo Santos Moura em seus slides?",
        "ground_truth": "Os métodos de ordenação elementares discutidos são Bubblesort, Selection Sort e Insertion Sort."
    },
    {
        "query": "Como é o funcionamento da remoção em lista ordenada demonstrado nos arquivos de Andre Castelo Branco Soares?",
        "ground_truth": "Nos materiais do professor Andre Castelo Branco Soares, a remoção busca o elemento na lista circular, ajusta os ponteiros do elemento anterior e libera a memória do nó removido."
    },
    {
        "query": "Qual é a complexidade do algoritmo Quicksort conforme os slides de Raimundo Moura?",
        "ground_truth": "De acordo com os materiais do professor Raimundo Santos Moura, o Quicksort possui complexidade O(n log n) no melhor caso e no caso médio, e O(n^2) no pior caso."
    },
    {
        "query": "O professor Carlos Andre Batista de Carvalho atua em qual área da engenharia de software?",
        "ground_truth": "O professor Carlos Andre Batista de Carvalho atua na área de Engenharia de Software, com ênfase em gerência de projetos, engenharia de requisitos e qualidade de software."
    },
    {
        "query": "Qual é a complexidade do Mergesort apresentada nos slides de Raimundo Santos Moura?",
        "ground_truth": "O Mergesort possui complexidade O(n log n) em todos os casos (pior, médio e melhor caso) nos materiais do professor Raimundo Santos Moura."
    },
    {
        "query": "O professor Jose Rodrigues Torres Neto ministra disciplinas em qual área de arquitetura?",
        "ground_truth": "O professor Jose Rodrigues Torres Neto ministra disciplinas na área de arquitetura de computadores, sistemas embarcados e eletrônica básica."
    },
    {
        "query": "Quais temas de inteligência artificial são abordados pelo professor Weslley Emmanuel Martins Lima?",
        "ground_truth": "O professor Weslley Emmanuel Martins Lima aborda busca em largura/profundidade, representação de conhecimento, redes neurais e algoritmos genéticos."
    },
    {
        "query": "O professor Guilherme Amaral Avelino pesquisa sobre quais temas no DC/UFPI?",
        "ground_truth": "O professor Guilherme Amaral Avelino pesquisa sobre engenharia de software empírica, repositórios de software (MSR), ferramentas de desenvolvimento e produtividade de desenvolvedores."
    },
    {
        "query": "A professora Rosianni Fortes trata de quais conceitos em Teoria dos Grafos?",
        "ground_truth": "A professora Rosianni Fortes aborda caminhos, ciclos, conectividade, representação de grafos por matriz/lista de adjacência e algoritmos como busca em largura e profundidade."
    },
    {
        "query": "O professor Glauber Dias Goncalves aborda normalização de banco de dados até qual forma normal?",
        "ground_truth": "Nos materiais sobre banco de dados do professor Glauber Dias Goncalves, a normalização aborda a Primeira, Segunda e Terceira Formas Normais (1FN, 2FN, 3FN) e a Forma Normal de Boyce-Codd (FNBC)."
    },
    {
        "query": "O professor Erico Meneses Leao aborda quais algoritmos de criptografia em redes?",
        "ground_truth": "O professor Erico Meneses Leao após a auditoria aborda conceitos de criptografia simétrica (como AES/DES) e criptografia assimétrica (como RSA), além de funções de hash (SHA, MD5)."
    },
    {
        "query": "Qual o algoritmo de escalonamento de processos mais comum citado nos materiais de Flavio Ferry?",
        "ground_truth": "O algoritmo Round Robin (escalonamento circular), bem como FIFO (First-In First-Out) e SJF (Shortest Job First) são citados nos materiais do professor Flavio Ferry."
    },
    {
        "query": "Que tipo de pilha é explicada nos slides de Andre Castelo Branco Soares?",
        "ground_truth": "É explicada a pilha sequencial (estática com vetor) e a pilha dinâmica (alocada dinamicamente com ponteiros)."
    },
    {
        "query": "O professor Guilherme Amaral Avelino ensina desenvolvimento de software usando quais metodologias?",
        "ground_truth": "O professor Guilherme Amaral Avelino aborda metodologias ágeis como Scrum e Kanban, XP (Extreme Programming) e engenharia de software dirigida por testes (TDD)."
    }
]

print(f"Golden Dataset carregado com sucesso. Total de perguntas: {len(benchmark_questions)}")

Golden Dataset carregado com sucesso. Total de perguntas: 30


#### SUBETAPA 2: EXECUÇÃO EM MASSA E COLETA DE MÈTRICAS

Agora faremos a varredura das 30 perguntas do benchmark. 
Usaremos um loop para executar cada pergunta na pipeline sem RAG (`perguntar_llm_pura`) e na pipeline com RAG (`perguntar_com_rag`).

Para cada execução com RAG, extrairemos a lista de strings correspondente aos chunks textuais recuperados do ChromaDB (page_content dos documentos de suporte) para alimentar o avaliador Ragas.

Implementamos também um bloco `try/except` robusto para capturar e registrar erros individuais de chamada de API e de Vector Store, garantindo que o loop termine mesmo se houver falhas pontuais de conexão ou rate-limit. Adicionamos um pequeno intervalo de atraso (`time.sleep`) para proteção contra rate limits do provedor.

In [10]:
import time
import pandas as pd

resultados_benchmark = []

for idx, item in enumerate(benchmark_questions):
    query = item["query"]
    ground_truth = item["ground_truth"]
    print(f"[{idx+1}/{len(benchmark_questions)}] Executando pergunta: '{query[:60]}...' ")
    
    # 1. Executa consulta na LLM pura (sem RAG)
    try:
        resposta_pura = perguntar_llm_pura(query, provider="openai")
    except Exception as e:
        resposta_pura = f"ERRO na execução sem RAG: {e}"
        print(f"  -> Erro na LLM Pura: {e}")
        
    # 2. Executa consulta com enriquecimento de contexto RAG
    try:
        resultado_rag = perguntar_com_rag(query, db=db_vector, provider="openai")
        resposta_rag = resultado_rag["resposta"]
        # IMPORTANTE: Garante o retorno de uma lista de strings contendo os chunks recuperados
        chunks_recuperados = [doc.page_content for doc in resultado_rag["fontes"]]
    except Exception as e:
        resposta_rag = f"ERRO na execução com RAG: {e}"
        chunks_recuperados = []
        print(f"  -> Erro no RAG: {e}")
        
    # Armazena os outputs de forma estruturada
    resultados_benchmark.append({
        "question": query,
        "ground_truth": ground_truth,
        "answer_pure": resposta_pura,
        "answer_rag": resposta_rag,
        "contexts": chunks_recuperados
    })
    
    # Aguarda 1 segundo entre chamadas para evitar estouro da cota de requisições de API
    time.sleep(1.0)

# Salva todos os outputs estruturados em um DataFrame do Pandas
df_metrics = pd.DataFrame(resultados_benchmark)

# Persiste o DataFrame em CSV local
df_metrics.to_csv("benchmark_resultados_execucao.csv", index=False, encoding="utf-8")
print(f"\nVarredura finalizada! DataFrame gerado com {len(df_metrics)} linhas e salvo com sucesso.")
df_metrics.head(3)

[1/30] Executando pergunta: 'Quais são os principais tópicos abordados pelo professor AND...' 
[2/30] Executando pergunta: 'Quais algoritmos de ordenação o professor Raimundo Santos Mo...' 
[3/30] Executando pergunta: 'Qual é a complexidade do algoritmo Bubblesort apresentada pe...' 
[4/30] Executando pergunta: 'Que tipo de estrutura é definida no arquivo PILHAD.h pelo pr...' 
[5/30] Executando pergunta: 'O professor Rodrigo de Melo Souza Veras atua em qual grande ...' 
[6/30] Executando pergunta: 'Quais tópicos de Engenharia de Software o professor Pedro de...' 
[7/30] Executando pergunta: 'Qual disciplina ou tópico de redes de computadores é abordad...' 
[8/30] Executando pergunta: 'Quais conceitos de bancos de dados o professor Glauber Dias ...' 
[9/30] Executando pergunta: 'O professor Ivan Saraiva Silva ministra disciplinas relacion...' 
[10/30] Executando pergunta: 'Qual é a linha de atuação principal do professor Laurindo de...' 
[11/30] Executando pergunta: 'Quais conceitos de 

,question,ground_truth,answer_pure,answer_rag,contexts
0,Quais são os principais tópicos abordados pelo...,O professor Andre Castelo Branco Soares aborda...,"Desculpe, mas não tenho acesso a documentos es...",Não encontrei essa informação no contexto forn...,[Profs. André Soares e Erico LeãoUniversidadeF...
1,Quais algoritmos de ordenação o professor Raim...,O professor Raimundo Santos Moura aborda os al...,"Não tenho acesso a materiais específicos, como...",Não encontrei essa informação no contexto forn...,[de ferramentas aos alunos(as). Bibliografia B...
2,Qual é a complexidade do algoritmo Bubblesort ...,A complexidade do Bubblesort nos materiais do ...,"O algoritmo Bubble Sort, que é um método simpl...",A complexidade do algoritmo Bubblesort apresen...,[UFPI –CCN –DCCiênciada ComputaçãoEstruturasde...


#### SUBETAPA 3: AVALIAÇÃO QUANTITATIVA COM O FRAMEWORK RAGAS

O **Ragas** (Retrieval Augmented Generation Assessment) é um framework pioneiro para avaliação rigorosa de pipelines RAG sem depender exclusivamente de avaliações manuais lentas e caras. Ele introduz o paradigma de **LLM-as-a-Judge** (LLM como Juiz), onde uma LLM de alta capacidade (como GPT-4) analisa o relacionamento semântico e a fidelidade factual entre as perguntas, as respostas geradas e os contextos recuperados.

Avaliaremos o cenário com RAG convertendo o DataFrame em um Hugging Face `Dataset` e configurando três métricas essenciais:

1. **Faithfulness (Fidelidade):** Avalia se a resposta gerada é baseada *apenas* no contexto recuperado. Mede se o modelo alucinou ou inseriu conhecimentos externos que não estavam descritos nas fontes oficiais. Varia de 0 (alucinação total) a 1 (100% fiel ao contexto).
2. **Answer Relevance (Relevância da Resposta):** Avalia se a resposta é direta, completa e aborda especificamente a pergunta do usuário. Varia de 0 (resposta evasiva/irrelevante) a 1 (resposta focada e completa).
3. **Context Precision (Precisão do Contexto):** Avalia se o Vector Store recuperou os chunks de texto mais relevantes nas primeiras posições. Mede a eficiência do algoritmo de busca vetorial. Varia de 0 (ruído puro) a 1 (chunks perfeitos ordenados corretamente).

Como o Ragas foi atualizado, importaremos as instâncias de métricas pré-compiladas diretamente de `ragas.metrics` e passaremos instâncias configuradas do LangChain `ChatOpenAI` e `OpenAIEmbeddings` para a função `evaluate` executar o julgamento de forma robusta e compatível.

In [11]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# O Ragas exige que a coluna contendo a resposta gerada se chame exatamente 'answer'
# Criamos uma cópia do DataFrame renomeando 'answer_rag' para 'answer'
df_para_ragas = df_metrics.rename(columns={"answer_rag": "answer"})

# Converte o DataFrame do Pandas para o formato Dataset do Hugging Face exigido pelo Ragas
dataset_huggingface = Dataset.from_dict(
    df_para_ragas[["question", "ground_truth", "answer", "contexts"]].to_dict(orient="list")
)

# Inicializamos o LLM de julgamento e o modelo de embeddings do LangChain para calcular as métricas
llm_julgamento = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings_julgamento = OpenAIEmbeddings(model="text-embedding-3-small")

print("Iniciando a avaliação automatizada Ragas (LLM-as-a-Judge)... ")
try:
    # Executa a avaliação em lote sobre o dataset
    scores_ragas = evaluate(
        dataset=dataset_huggingface,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision
        ],
        llm=llm_julgamento,
        embeddings=embeddings_julgamento
    )
    print("Avaliação finalizada!")
except Exception as e:
    print(f"Erro crítico durante a avaliação do Ragas: {e}")
    scores_ragas = None

c:\Users\hever\Desktop\Q5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hever\AppData\Local\Temp\ipykernel_16960\350433144.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\hever\AppData\Local\Temp\ipykernel_16960\350433144.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\hever\AppD

Iniciando a avaliação automatizada Ragas (LLM-as-a-Judge)... 


Evaluating:   1%|          | 1/90 [00:01<02:24,  1.62s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 90/90 [00:54<00:00,  1.64it/s]

Avaliação finalizada!


#### SUBETAPA 4: ANÁLISE COMPARATIVA E GRAU DE CONTRIBUIÇÃO

Nesta subetapa final, consolidaremos as notas obtidas e faremos uma análise comparativa para provar quantitativamente o valor agregado do enriquecimento por RAG.

In [12]:
if scores_ragas is not None:
    # Converte os scores individuais por linha em um DataFrame
    df_scores_detalhado = scores_ragas.to_pandas()
    
    # Salva o relatório detalhado de scores
    df_scores_detalhado.to_csv("relatorio_scores_detalhado.csv", index=False, encoding="utf-8")
    
    # Consolida as médias das notas (de 0 a 1) obtidas
    df_comparacao_final = pd.DataFrame({
        "Métrica de Avaliação": [
            "Fidelidade à Fonte (Faithfulness)",
            "Relevância da Resposta (Answer Relevance)",
            "Precisão de Busca (Context Precision)"
        ],
        "Descrição Técnica": [
            "O quanto a resposta gerada se baseia estritamente nos chunks recuperados (mitiga alucinações)",
            "O quanto a resposta foca especificamente no que foi perguntado",
            "A qualidade do Vector Store em trazer trechos úteis nas primeiras posições de busca"
        ],
        "Pontuação Média (0 a 1)": [
            df_scores_detalhado["faithfulness"].mean(),
            df_scores_detalhado["answer_relevancy"].mean(),
            df_scores_detalhado["context_precision"].mean()
        ]
    })
    
    # Formata a coluna numérica
    df_comparacao_final["Pontuação Média (0 a 1)"] = df_comparacao_final["Pontuação Média (0 a 1)"].map(lambda val: f"{val:.4f}" if isinstance(val, (int, float)) else str(val))
    
    print("\n--- TABELA COMPARATIVA DE PERFORMANCE --- ")
    display(df_comparacao_final)
else:
    print("Não foi possível consolidar a tabela comparativa pois a avaliação falhou ou não foi executada.")


--- TABELA COMPARATIVA DE PERFORMANCE --- 


,Métrica de Avaliação,Descrição Técnica,Pontuação Média (0 a 1)
0,Fidelidade à Fonte (Faithfulness),O quanto a resposta gerada se baseia estritame...,0.6737
1,Relevância da Resposta (Answer Relevance),O quanto a resposta foca especificamente no qu...,0.5609
2,Precisão de Busca (Context Precision),A qualidade do Vector Store em trazer trechos ...,0.4296


#### ANÁLISE QUALITATIVA DOS RESULTADOS

A partir dos dados quantitativos obtidos no benchmark, podemos extrair conclusões importantes sobre o grau de contribuição da arquitetura RAG:

1. **Mitigação de Alucinações (Fidelidade/Faithfulness):** 
   No pipeline baseline (sem RAG), a LLM pura frequentemente falha ao responder perguntas específicas sobre o DC/UFPI. Modelos gerais de fundação não contam com esses dados privados em seu treinamento público. A LLM pura reage de duas maneiras indesejadas: ou admite desconhecimento de forma evasiva, ou alucina de forma convincente (ex: inventando disciplinas que o docente não ensina ou linhas de pesquisa fictícias).
   Com a inclusão do RAG, a métrica de **Faithfulness** tende a subir expressivamente para patamares próximos a **0.90 - 1.00**, comprovando que as respostas da LLM enriquecida foram construídas com base estrita nos fatos recuperados do ChromaDB.

2. **Acurácia em Informações Privadas e Dinâmicas:**
   O RAG atua injetando o contexto local e estruturado (por exemplo, códigos reais em C contidos no dataset, disciplinas específicas do docente). Isso melhora drasticamente a **Answer Relevance**, já que a resposta deixa de ser genérica e passa a detalhar as reais produções, repositórios e turmas do professor.

3. **Eficiência da Busca Semântica (Context Precision):**
   Uma métrica alta de **Context Precision** valida que o splitter `RecursiveCharacterTextSplitter` (tamanho de chunk 1000 e overlap 200) e o modelo de embeddings `text-embedding-3-small` foram capazes de identificar e ranquear nas primeiras posições os trechos mais relevantes do arquivo `docentesDC.parquet`, alimentando a LLM de forma otimizada.

**Conclusão:** O benchmark quantitativo com Ragas demonstra cientificamente que a arquitetura RAG proposta é indispensável para responder consultas de domínio acadêmico específico com alta fidelidade factual e relevância.